In [1]:
from stark_qa import load_skb
import regex as re
import ast
import torch
import vss
from stark_qa import load_qa
import sys
from urllib import response
import os
import os
from dotenv import load_dotenv

load_dotenv(override=True)
from custom_pipeline.entity_parsing import *
from custom_pipeline.relation_parsing import *
from custom_pipeline.llm_bridge import LlmBridge
# print("OPENAI_API_KEY =", os.getenv("OPENAI_API_KEY"))
# print("NVIDIA_API_KEY =", os.getenv("NVIDIA_API_KEY"))


OpenAI API key found.


In [2]:
data_split = 'test'

emb_model = 'text-embedding-ada-002'
configs_path = 'configs.json'
qa_dataset = load_qa('prime')
qa = qa_dataset.split_indices[data_split].reshape(-1).tolist()
# qa = qa[:int(len(qa) * 0.1)]
test_queries = [qa_dataset[i] for i in qa]

qa_dataset[6517]

dataset_name = 'prime'
kb = load_skb(dataset_name, download_processed=True)



# model_name = "openai/gpt-oss-120b"  # or "deepseek-chat", "gemini-pro", etc.
model_name = 'meta/llama-3.3-70b-instruct' 

# Optional: Initialize logger if you want logging
# logger = Logger("temp/")  # Adjust based on your Logger implementation

# Create LLM Bridge instance
llm_bridge = LlmBridge(model_name=model_name,configs_path="configs.json")

# framework = Framework("cypher_notebook", dataset_name, data_split, llm_model=llm_model, enable_vss=True,
#                   emb_model=emb_model, configs_path=configs_path)
# alpha = framework.settings.get("alpha")

nodes_to_consider = str(kb.node_type_lst()).replace("'","")
# edges_to_consider = str(list(framework.settings.configs.get("edge_type_long2short").keys())).replace("'","")
# properties_to_consider = str(list(framework.settings.configs.get("avail_node_properties").keys())).replace("'","")
relation_names = ['ppi', 'carrier', 'enzyme', 'target', 'transporter', 'contraindication', 'indication', 'off-label use', 'synergistic interaction', 'associated with', 'parent-child', 'phenotype absent', 'phenotype present', 'side effect', 'interacts with', 'linked to', 'expression present', 'expression absent']

Use file from /home/sarthak/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/qa/prime/stark_qa/stark_qa_human_generated_eval.csv.
Loading from /home/sarthak/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/skb/prime/processed!
[LlmBridge] Initializing model: meta/llama-3.3-70b-instruct (device=cpu)
[LlmBridge] Using NVIDIA/meta-style model: meta/llama-3.3-70b-instruct


In [3]:

from typing import List, Dict, Any
class Query:
    """
    A class to hold the attributes of a query and its processing state.
    """    
    def __init__(
        self, 
        id: Any, 
        query: str, 
        ground_truths: List[Any],
        status: str = "IN_PROGRESS",
        prompt: str = "",
        entities: Dict[str, Any] = None,
        symbol_candidates: Dict[str, Any] = None,
        relations : Dict[tuple, Any] = None ,
    ):
        self.id = id
        self.query = query
        self.ground_truths = ground_truths
        self.status = status
        self.entity_id_response = ""
        self.relations_id_response = ""
        self.relations = relations        
        # Use None as default for mutable types (dicts/lists)
        # and initialize to empty dict if None is passed.
        self.entities = entities if entities is not None else {}
        self.initial_symbol_candidates = symbol_candidates if symbol_candidates is not None else {}
        self.relations = entities if entities is not None else {}
        self.results = {}

    def __repr__(self) -> str:
        """Provides a clean string representation for the object."""
        return (f"Query(id={self.id!r}, query={self.query!r}, "
                f"status={self.status!r}, ground_truths={self.ground_truths!r})")

In [4]:
# Step 1 : getting Entities from LLM

def generate_entity_identification_prompt(query_str) :
    with open("./prompts/entity_identification.txt","r") as file :
        prompt_prefix = file.read()
        return prompt_prefix + query_str
    
def get_llm_response(prompt) :
    response = llm_bridge.ask_llm_batch([prompt])
    print(response)
    return response[0][0]

def step1_identify_entities(query: Query) :
    prompt = generate_entity_identification_prompt(query.query)
    response_string = get_llm_response(prompt)
    query.entity_id_response = response_string
    if response_string == '' :
        query.status = "FAILED"
        return # should i be doing this ??
    else :
        try :
            identified_entities = parse_entity_response(response_string)
            query.entities = identified_entities
        except ValueError as e:
            query.status = "FAILED"
            print(f"Error parsing response for query {e}")

### Step 2 : identifying the relations between entities 


def generate_relation_identification_prompt(query_string,entities_string) :
    with open("./prompts/relations_identification.txt","r") as file :
        prompt_prefix = file.read()
        return prompt_prefix + f" Q : {query_string} " + f"Identified Entities : {entities_string}"
    
    
def step2_identify_relations(query: Query) :
    prompt = generate_relation_identification_prompt(query.query,str(query.entities))
    response_string = get_llm_response(prompt)
    query.relations_id_response = response_string

    if response_string == '' :
        query.status = "FAILED"
        return # should i be doing this ??
    else :
        try :
            identified_relations = parse_relation_string(response_string)
            query.relations = identified_relations
        except ValueError as e:
            query.status = "FAILED"
            print(f"Error parsing response for query {e}")

In [5]:
## intermediate step : initializing VSS (fucntion definitions and all)

## Step 3 : getting initial candidates for all the symbols except answer 
from custom_pipeline.candidate_context import CandidateContext
from custom_pipeline.vss_retreiver import VSSRetriever

# Initialize the retriever
retriever = VSSRetriever(
    kb=kb,
    emb_base_path="./emb/prime",
    emb_model="text-embedding-ada-002",
    qa_dataset=qa_dataset,
    dataset_name="test",
    use_vss=True  # Set to True if you need VSS object for generating new embeddings
)

# Now you can use the retriever
print("Available node types:", retriever.get_available_node_types())
# Retrieve top k nodes
top_nodes, scores = retriever.get_top_k_nodes(
    search_str="your query here",
    k=10,
    node_type="drug",
    cutoff=0.5
)

# Access the loaded embeddings if needed
entity_emb_dict = retriever.entity_emb_dict
query_emb_dict = retriever.query_emb_dict
candidate_emb_dict = retriever.candidate_emb_dict
node_emb_dict = retriever.node_emb_dict

Loading query embeddings from emb/prime/text-embedding-ada-002/query/query_emb_dict.pt.
Loaded 11204 query embeddings from emb/prime/text-embedding-ada-002/query/query_emb_dict.pt!
Loaded embeddings of nodes from emb/prime/text-embedding-ada-002/nodes!
Loaded 684 entity embeddings from emb/prime/text-embedding-ada-002/entities/entity_emb_dict.pt!
Available node types: ['biological_process', 'exposure', 'gene/protein', 'drug', 'pathway', 'effect/phenotype', 'anatomy', 'molecular_function', 'cellular_component', 'disease']
Getting embedding for query: your query here using model: text-embedding-ada-002


In [6]:
def get_initial_candidates_for_entity(entity_info, entity_key, kb, retriever):
    print(entity_info)
    # entity_info = entity_info[entity_key]
    candidates = []
    entity_types = entity_info.get("type", [])
    name_constraint = entity_info.get("lexical", {}).get("name", None)
    semantic_constraints = entity_info.get("semantic", []).copy()
    # if len(semantic_constraints) == 0:   ## doing this because there was one node that had some phrase "a b c" , "a b" was identified as name, only c didn't make sense as lexical constraint
    #     semantic_constraints = []
    for key in entity_info.get("lexical", {}):
        semantic_constraints.append(f" {entity_info['lexical'][key]}")
   
    nodes_by_name = []

    if name_constraint:
        for etype in entity_types:
            print("Searching for entity type:", etype)
            nodes_by_name = kb.get_node_ids_by_value(node_type=etype, key="name", value=name_constraint)
            print("Found nodes by name:", nodes_by_name)
            candidates.extend([CandidateContext(node_id=x, entity=entity_key,score=1)for x in nodes_by_name])

    vss_candidates_count = 25 - len(candidates) # total 25 candidates instead of 20 

    for etype in entity_types : 
        sem = "".join(semantic_constraints)
        print("Searching for semantic constraint:", sem)
        nodes_by_desc_ids , vss_scores = retriever.get_top_k_nodes(search_str=sem, k=vss_candidates_count, node_type=etype, cutoff=0.65) # adding a cutoff
        candidates.extend([CandidateContext(node_id=nodes_by_desc_ids[x], entity=entity_key,score=vss_scores[x])for x in range(len(nodes_by_desc_ids)) if nodes_by_desc_ids[x] not in nodes_by_name])
        print("Found nodes by description:", len(nodes_by_desc_ids))
            # candidates.extend(nodes_by_desc)
    print("Candidates so far:", candidates)
    return candidates # Remove duplicates

def step3_get_initial_candidates(current_query) : 
    initial_candidates = {}
    print(initial_candidates)
    for entity in current_query.entities:
        print("Entity:", entity)
        if entity == "ANSWER":
            continue
        initial_candidates[entity] = get_initial_candidates_for_entity(current_query.entities[entity],entity_key=entity,kb=kb,retriever=retriever)
    current_query.initial_symbol_candidates = initial_candidates


In [7]:
# Step 4 : Grounding

from custom_pipeline.grounders.grounder3 import PriorityQueueGrounding
# print(dir(PriorityQueueGrounding))  # Should show 'ground' in the list
import heapq
from typing import Dict, Set, List, Tuple
from collections import defaultdict
from custom_pipeline.candidate_context import CandidateContext
from custom_pipeline.grounders.grounder3 import PriorityQueueGrounding


def run_priority_queue_grounding(
    query_obj,
    kb,
    vss_retriever,
    max_candidates_per_symbol: int = 1000,
    max_answer_candidates: int = 50,
    top_k_neighbors: int = 10,
    score_decay: float = 0.9,
    support_boost : float = 0.15 ,
    verbose: bool = False
) -> Dict[str, List[CandidateContext]]:
    """
    Convenience function to run priority queue-based grounding.
    
    Args:
        query_obj: Query class instance with entities, relations, and initial_symbol_candidates
        kb: Knowledge base object
        vss_retriever: VSSRetriever instance for scoring neighbors
        max_candidates_per_symbol: Max candidates per entity (default 1000)
        max_answer_candidates: Max candidates for ANSWER entity (default 100)
        top_k_neighbors: Number of top neighbors to add per expansion (default 10)
        score_decay: Score decay factor for propagation (default 0.9)
        verbose: Print detailed logs (default True)
    
    Returns:
        Dictionary mapping entity symbols to lists of CandidateContext objects
    """
    grounder = PriorityQueueGrounding(
        query_obj=query_obj,
        kb=kb,
        vss_retriever=vss_retriever,
        max_candidates_per_symbol=max_candidates_per_symbol,
        max_answer_candidates=max_answer_candidates,
        top_k_neighbors=top_k_neighbors,
        score_decay=score_decay,
        support_boost=support_boost,
        verbose=verbose
    )
    
    return grounder.ground()

def evaluate_result(predicted_nodes,ground_truth_nodes) : 
    
    # Convert ground truth to set for O(1) lookup
    ground_truth_set = set(ground_truth_nodes)
    
    # Handle empty predictions - THIS MUST BE FIRST
    if not predicted_nodes:
        return {
            "answer_set": set(),
            "ground_truth_set": ground_truth_set,
            "retrieved_ground_truths": set(),
            "missed_ground_truths": ground_truth_set,
            "metrics": {
                "total_answers": len(ground_truth_set),
                "retrieved_count": 0,
                "missed_count": len(ground_truth_set),
                "recall@50": 0.0,
                "hit_at_1": 0.0,
                "hit_at_5": 0.0,
                "mrr": 0.0,
            }
        }
    
    # Hit@1: Check if the first predicted node is in ground truth
    hit_at_1 = 1.0 if predicted_nodes[0] in ground_truth_set else 0.0
    
    # Hit@5: Check if any of the first 5 predicted nodes are in ground truth
    hit_at_5 = 1.0 if any(node in ground_truth_set for node in predicted_nodes[:5]) else 0.0
    
    # MRR (Mean Reciprocal Rank): Find the rank of the first relevant item
    mrr = 0.0
    for rank, node in enumerate(predicted_nodes, 1):
        if node in ground_truth_set:
            mrr = 1.0 / rank
            break
    answer_set = set(predicted_nodes)
    retieved_ground_truths = answer_set.intersection(ground_truth_set)
    missed_ground_truths = ground_truth_set.difference(retieved_ground_truths)
    retieved_ground_truths_count = len(retieved_ground_truths)
    missed_ground_truths_count = len(missed_ground_truths)
    recall = retieved_ground_truths_count / len(ground_truth_set) if len(ground_truth_set) > 0 else 0.0
        # Recall@20
    top_20_predictions = predicted_nodes[:20]
    retrieved_20 = set(top_20_predictions).intersection(ground_truth_set)
    recall_20 = len(retrieved_20) / len(ground_truth_set) if len(ground_truth_set) > 0 else 0.0

    # Store all results in a dictionary
    results = {
        # "answer_set": answer_set,
        "answer_list": predicted_nodes,
        "ground_truth_set": ground_truth_set,
        "retrieved_ground_truths": retieved_ground_truths,
        "missed_ground_truths": missed_ground_truths,
        "metrics": {
        "total_answers" : len(ground_truth_set),
        "retrieved_count": retieved_ground_truths_count,
        "missed_count": missed_ground_truths_count,
        "recall@50": recall,        
        "recall@20" : recall_20 , 
        'hit_at_1': hit_at_1,
        'hit_at_5': hit_at_5,
        'mrr': mrr,
        }


    }
    return results

In [8]:
def step4_grounding(
    query: Query,
    max_candidates_per_symbol: int = 1000,
    max_answer_candidates: int = 50,
    top_k_neighbors: int = 10,
    score_decay: float = 0.9,
    support_boost : float = 0.15 ,
    verbose: bool = False
):
    final_candidates = run_priority_queue_grounding(
        query_obj=query,  # Your Query instance
        kb=kb,
        vss_retriever=retriever,
        max_candidates_per_symbol=max_candidates_per_symbol, # increasing max canidates per symbol to 3000
        max_answer_candidates=max_answer_candidates,
        top_k_neighbors=top_k_neighbors,
        support_boost=support_boost,   # increasing support boost 
        score_decay=score_decay,
        verbose=verbose
    )
    answers = [cc.node_id for cc in  sorted(final_candidates["ANSWER"], key = lambda x : (x.score , - x.support), reverse= True)  ]
    # reranking candidates using support, followed by score.
    results = evaluate_result(answers,query.ground_truths)
    query.results = results
    return final_candidates




In [9]:
## saving results 
import csv
from pathlib import Path
from typing import Optional

def save_results(
    query: Query, 
    csv_path: str = "pipeline_results.csv",
    append: bool = True
):
    """
    Save query results metrics to a CSV file.
    
    Args:
        query: Query object with results populated
        csv_path: Path to the CSV file (default: "pipeline_results.csv")
        append: If True, append to existing file; if False, overwrite (default: True)
    """
    csv_file = Path(csv_path)
    
    # Check if file exists and if we should write headers
    file_exists = csv_file.exists()
    write_headers = not file_exists or not append
    
    # Open file in append or write mode
    mode = 'a' if append else 'w'
    
    # Extract metrics from query results
    if not hasattr(query, 'results') or query.results is None:
        print(f"[WARNING] Query {query.id} has no results. Skipping.")
        return
    
    metrics = query.results.get('metrics', {})
    
    # Prepare row data
    row_data = {
        'query_id': query.id,
        'query_text': query.query,
        'total_answers': metrics.get('total_answers', 0),
        'retrieved_count': metrics.get('retrieved_count', 0),
        'missed_count': metrics.get('missed_count', 0),
        'recall@20': metrics.get('recall@20', 0.0),
        'recall@50': metrics.get('recall@50', 0.0),
        'hit@1': metrics.get('hit_at_1', 0.0),
        'hit@5': metrics.get('hit_at_5', 0.0),
        'mrr': metrics.get('mrr', 0.0),
    }
    
    # Define fieldnames (column order)
    fieldnames = [
        'query_id',
        'query_text',
        'total_answers',
        'retrieved_count',
        'missed_count',
        'recall@50',
        'recall@20',
        'hit@1',
        'hit@5',
        'mrr'
    ]
    
    with open(csv_file, mode, newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        
        # Write header if needed
        if write_headers:
            writer.writeheader()
        
        # Write the data row
        writer.writerow(row_data)
    
    print(f"[SAVED] Results for query {query.id} saved to {csv_path}")


def save_aggregate_results(
    queries: list,
    csv_path: str = "aggregate_results.csv"
):
    """
    Calculate and save aggregate metrics across all queries.
    
    Args:
        queries: List of Query objects with results
        csv_path: Path to save aggregate results
    """
    if not queries:
        print("[WARNING] No queries provided for aggregation.")
        return
    
    # Collect all metrics
    all_metrics = []
    for query in queries:
        if hasattr(query, 'results') and query.results is not None:
            all_metrics.append(query.results.get('metrics', {}))
    
    if not all_metrics:
        print("[WARNING] No valid results found in queries.")
        return
    
    # Calculate averages
    num_queries = len(all_metrics)
    avg_metrics = {
        'total_queries': num_queries,
        'avg_total_answers': sum(m.get('total_answers', 0) for m in all_metrics) / num_queries,
        'avg_retrieved_count': sum(m.get('retrieved_count', 0) for m in all_metrics) / num_queries,
        'avg_missed_count': sum(m.get('missed_count', 0) for m in all_metrics) / num_queries,
        'avg_recall@20': sum(m.get('recall@20', 0.0) for m in all_metrics) / num_queries,
        'avg_recall@50': sum(m.get('recall@50', 0.0) for m in all_metrics) / num_queries,
        'avg_hit@1': sum(m.get('hit_at_1', 0.0) for m in all_metrics) / num_queries,
        'avg_hit@5': sum(m.get('hit_at_5', 0.0) for m in all_metrics) / num_queries,
        'avg_mrr': sum(m.get('mrr', 0.0) for m in all_metrics) / num_queries,
    }
    
    # Write to CSV
    fieldnames = list(avg_metrics.keys())
    
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerow(avg_metrics)
    
    print(f"[SAVED] Aggregate results saved to {csv_path}")
    print(f"\nAggregate Metrics:")
    for key, value in avg_metrics.items():
        print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")



In [10]:

qa_dataset = load_qa('prime')
qa = qa_dataset.split_indices[data_split].reshape(-1).tolist()
qa = qa[:int(len(qa) * 0.1)]
# qa = [7152, 5659, 3478, 9020, 9257, 6522, 181, 4045, 1118]
test_queries = [qa_dataset[i] for i in qa]

Use file from /home/sarthak/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/qa/prime/stark_qa/stark_qa_human_generated_eval.csv.


In [11]:
import pandas as pd
df = pd.read_csv('./STEP2_results/full_data_dump.csv',header=0)

queries = []

import json
import ast
import math
import re

def parse_mixed(val):
    # Already parsed dict
    if isinstance(val, dict):
        return val

    # NaN or None
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return {}

    s = str(val).strip()

    # Remove one outer layer of quotes
    if (s.startswith('"') and s.endswith('"')) or (s.startswith("'") and s.endswith("'")):
        s = s[1:-1].strip()

    # ---- Try JSON first ----
    try:
        return json.loads(s)
    except:
        pass

    # ---- Convert JSON tokens -> Python tokens ----
    # true -> True, false -> False, null -> None
    fixed = (
        s.replace("true", "True")
         .replace("false", "False")
         .replace("null", "None")
    )

    # ---- Try Python literal ----
    try:
        return ast.literal_eval(fixed)
    except:
        print("UNPARSABLE STRING:", repr(s))
        raise

stored_data = {}
for index, query in df.iterrows():
    # print(query)
    stored_data[query['id']] = {
        'entities' : (query['entity_id_response']),
        'relations' : (query['relations_id_response'])
    }

print(df.head())
def dummy_step1_identify_entities(current_query):
    response_string = ""
    if current_query.id in stored_data :
        response_string = stored_data[current_query.id]['entities']
        current_query.entity_id_response = response_string

    else :
        response_string = ''
    
    if response_string == '':
        current_query.status = "FAILED"
        return  # should i be doing this ??
    else:
        try:
            identified_entities = parse_entity_response(response_string)
            current_query.entities = identified_entities
        except ValueError as e:
            current_query.status = "FAILED"
            print(f"Error parsing response for query {e}")

# def step1_identify_entities(query: Query) :
    
#     query.entity_id_response = response_string
#     if response_string == '' :
#         query.status = "FAILED"
#         return # should i be doing this ??
#     else :
#         try :
#             identified_entities = parse_entity_response(response_string)
#             query.entities = identified_entities
#         except ValueError as e:
#             query.status = "FAILED"
#             print(f"Error parsing response for query {e}")

def dummy_step2_identify_relations(current_query) : 
    response_string = ""
    if current_query.id in stored_data :
        response_string = stored_data[current_query.id]['relations']
    else :
        response_string = ''
    current_query.relations_id_response = response_string
    if response_string == '' :
        current_query.status = "FAILED"
        return # should i be doing this ??
    else :
        try :
            identified_relations = parse_relation_string(response_string)
            current_query.relations = identified_relations
        except ValueError as e:
            current_query.status = "FAILED"
            print(f"Error parsing response for query {e}")

# def step2_identify_relations(query: Query) :
    # prompt = generate_relation_identification_prompt(query.query,str(query.entities))
    # response_string = get_llm_response(prompt)
    # query.relations_id_response = response_string

    # if response_string == '' :
    #     query.status = "FAILED"
    #     return # should i be doing this ??
    # else :
    #     try :
    #         identified_relations = parse_relation_string(response_string)
    #         query.relations = identified_relations
    #     except ValueError as e:
    #         query.status = "FAILED"
    #         print(f"Error parsing response for query {e}")

      id                                              query  \
0   6517  Which cell structures are involved in interact...   
1   9996  Which genes or proteins are expressed exclusiv...   
2   3630  Could you suggest any medications effective fo...   
3  10252  Which gene or protein responsible for coding t...   
4   5142  Which autosomal dominant diseases are linked t...   

                  ground_truths       status  \
0  [56241, 55842, 56174, 56263]  IN_PROGRESS   
1                       [10833]  IN_PROGRESS   
2                       [15179]  IN_PROGRESS   
3                    [43, 2140]  IN_PROGRESS   
4                       [30638]  IN_PROGRESS   

                                  entity_id_response  \
0  {\n  "A": {\n    "type": ["drug"],\n    "lexic...   
1  {\n  "A": {\n    "type": ["anatomy"],\n    "le...   
2  {\n  "A": {\n    "type": ["disease"],\n    "le...   
3  {\n  "A": {\n    "type": ["gene/protein", "mol...   
4  {\n  "A": {\n    "type": ["anatomy"],\n    "le...

In [12]:
def pipeline(query,exp_name,max_candidates_per_symbol,max_answer_candidates,top_k_neighbors,score_decay,support_boost) :
    dummy_step1_identify_entities(query)
    dummy_step2_identify_relations(query)
    print(query)
    step3_get_initial_candidates(query)
    step4_grounding(query,max_candidates_per_symbol=max_candidates_per_symbol,max_answer_candidates=max_answer_candidates,top_k_neighbors=top_k_neighbors,score_decay=score_decay,support_boost=support_boost,verbose=False)
    save_results(query, csv_path=f"./output/{exp_name}/pipeline_results.csv")


In [18]:
import os
import csv
import json
import itertools
import contextlib
import io
import traceback
import signal
from tqdm import tqdm
from collections import Counter

# ==========================================
# 1. SETUP & HELPER FUNCTIONS
# ==========================================

def timeout_handler(signum, frame):
    raise TimeoutError("Query execution timed out")

# Register signal for timeout
signal.signal(signal.SIGALRM, timeout_handler)
TIMEOUT_SECONDS = 60  # 5 minutes

def save_data_dump(results, csv_path="aggregate_results.csv"):
    """Saves Query objects to a CSV file handling dict/list serialization."""
    if not results: return
    
    def serialize_value(value):
        if value is None: return ""
        if isinstance(value, (str, int, float, bool)): return value
        if isinstance(value, dict): return {k: serialize_value(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)): return [serialize_value(item) for item in value]
        return str(value)

    # dynamically get fields from the first result or a fixed list
    fieldnames = ['id', 'query', 'ground_truths', 'status', 'entity_id_response', 
                  'relations_id_response', 'entities', 'initial_symbol_candidates', 
                  'relations', 'results']
    
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for query_obj in results:
            row = {}
            for field in fieldnames:
                value = getattr(query_obj, field, "")
                if value is None: row[field] = ""
                elif isinstance(value, (dict, list, tuple)):
                    try: row[field] = json.dumps(serialize_value(value))
                    except: row[field] = str(value)
                else: row[field] = str(value)
            writer.writerow(row)

# ==========================================
# 2. HYPERPARAMETER GRID
# ==========================================

# Define the lists of parameters you want to try
# The code will run the pipeline for EVERY combination of these parameters
param_grid = {
    'max_candidates_per_symbol': [500,3000],
    'max_answer_candidates': [20],
    'top_k_neighbors': [10,15,20],
    'score_decay': [0.95,0.85],
    'support_boost': [0,0.1,0.15]
}

# Generate all combinations
keys, values = zip(*param_grid.items())
param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"🚀 Found {len(param_combinations)} unique parameter combinations to run.")

# ==========================================
# 3. EXECUTION LOOP
# ==========================================

for i, params in enumerate(param_combinations):
    
    # 3.1 Create Experiment Name and Directory
    # Create a compact name like: exp_MC5_MA10_K50_SD0.5_SB0.1
    param_str = "_".join([f"{k.split('_')[0][:3].upper()}{v}" for k, v in params.items()]) 
    exp_name = f"GROUNDER3{i+1}_{param_str}"
    
    output_dir = f"./output/{exp_name}"
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n==================================================")
    print(f"▶ Starting Experiment {i+1}/{len(param_combinations)}: {exp_name}")
    print(f"Parameters: {params}")
    print(f"Output Directory: {output_dir}")
    print(f"==================================================")

    failed_queries = []
    results = []

    # 3.2 Run Queries for this Parameter Set
    # We copy test_queries to avoid modifying the original list if necessary
    for query_data in tqdm(test_queries, desc=f"Exp {i+1} Queries", unit="query"):
        
        # Extract query data
        q_text, q_id, q_gt = query_data[0], query_data[1], query_data[2]
        
        try:
            # Create a clean buffer to capture stdout/stderr to avoid cluttering notebook
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                
                # Re-instantiate the query object for this run
                current_query = Query(id=q_id, query=q_text, ground_truths=q_gt) 
                
                # Set Timeout Alarm
                signal.alarm(TIMEOUT_SECONDS)
                try:
                    # PASS PARAMETERS TO PIPELINE
                    pipeline(
                        query=current_query,
                        exp_name=exp_name,
                        **params  # Unpack the current dictionary of parameters
                    )
                finally:
                    signal.alarm(0) # Disable alarm immediately
            
            # If successful
            results.append(current_query)

        except TimeoutError as e:
            failed_queries.append({
                'query_id': q_id,
                'query_text': q_text[:100],
                'error': f"Timeout after {TIMEOUT_SECONDS}s",
                'error_type': 'TimeoutError',
                'traceback': 'Signal Alarm Triggered'
            })
            # tqdm.write(f"✗ Query {q_id} timed out.")
            
        except Exception as e:
            signal.alarm(0) # Ensure alarm is off
            failed_queries.append({
                'query_id': q_id,
                'query_text': q_text[:100],
                'error': str(e),
                'error_type': type(e).__name__,
                'traceback': traceback.format_exc()
            })
            # tqdm.write(f"✗ Query {q_id} failed: {type(e).__name__}")

    # ==========================================
    # 4. SAVE RESULTS FOR THIS EXPERIMENT
    # ==========================================
    
    # Save Full Data Dump
    dump_path = os.path.join(output_dir, "full_data_dump.csv")
    save_data_dump(results, csv_path=dump_path)
    save_aggregate_results(results, csv_path=os.path.join(output_dir, "aggregate_results.csv"))
    # Save Failed Queries
    if failed_queries:
        fail_path = os.path.join(output_dir, "failed_queries.csv")
        with open(fail_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=['query_id', 'query_text', 'error_type', 'error', 'traceback'])
            writer.writeheader()
            writer.writerows(failed_queries)
        
        error_counts = Counter(fq['error_type'] for fq in failed_queries)
        print(f"⚠ {len(failed_queries)} queries failed. Top errors: {dict(error_counts)}")
    else:
        print(f"✓ All {len(results)} queries processed successfully.")

print("\n🎉 All experiments completed.")

🚀 Found 36 unique parameter combinations to run.

▶ Starting Experiment 1/36: GROUNDER31_MAX500_MAX20_TOP10_SCO0.95_SUP0
Parameters: {'max_candidates_per_symbol': 500, 'max_answer_candidates': 20, 'top_k_neighbors': 10, 'score_decay': 0.95, 'support_boost': 0}
Output Directory: ./output/GROUNDER31_MAX500_MAX20_TOP10_SCO0.95_SUP0


Exp 1 Queries: 100%|██████████| 280/280 [08:26<00:00,  1.81s/query]


[SAVED] Aggregate results saved to ./output/GROUNDER31_MAX500_MAX20_TOP10_SCO0.95_SUP0/aggregate_results.csv

Aggregate Metrics:
  total_queries: 277
  avg_total_answers: 2.5126
  avg_retrieved_count: 1.3141
  avg_missed_count: 1.1986
  avg_recall@20: 0.6039
  avg_recall@50: 0.6039
  avg_hit@1: 0.3394
  avg_hit@5: 0.5271
  avg_mrr: 0.4212
⚠ 3 queries failed. Top errors: {'TimeoutError': 3}

▶ Starting Experiment 2/36: GROUNDER32_MAX500_MAX20_TOP10_SCO0.95_SUP0.1
Parameters: {'max_candidates_per_symbol': 500, 'max_answer_candidates': 20, 'top_k_neighbors': 10, 'score_decay': 0.95, 'support_boost': 0.1}
Output Directory: ./output/GROUNDER32_MAX500_MAX20_TOP10_SCO0.95_SUP0.1


Exp 2 Queries: 100%|██████████| 280/280 [08:22<00:00,  1.79s/query]


[SAVED] Aggregate results saved to ./output/GROUNDER32_MAX500_MAX20_TOP10_SCO0.95_SUP0.1/aggregate_results.csv

Aggregate Metrics:
  total_queries: 277
  avg_total_answers: 2.5126
  avg_retrieved_count: 1.3141
  avg_missed_count: 1.1986
  avg_recall@20: 0.6039
  avg_recall@50: 0.6039
  avg_hit@1: 0.3249
  avg_hit@5: 0.5090
  avg_mrr: 0.4087
⚠ 3 queries failed. Top errors: {'TimeoutError': 3}

▶ Starting Experiment 3/36: GROUNDER33_MAX500_MAX20_TOP10_SCO0.95_SUP0.15
Parameters: {'max_candidates_per_symbol': 500, 'max_answer_candidates': 20, 'top_k_neighbors': 10, 'score_decay': 0.95, 'support_boost': 0.15}
Output Directory: ./output/GROUNDER33_MAX500_MAX20_TOP10_SCO0.95_SUP0.15


...

[SAVED] Aggregate results saved to ./output/GROUNDER334_MAX3000_MAX20_TOP20_SCO0.85_SUP0/aggregate_results.csv

Aggregate Metrics:
  total_queries: 275
  avg_total_answers: 2.5200
  avg_retrieved_count: 1.4218
  avg_missed_count: 1.0982
  avg_recall@20: 0.6225
  avg_recall@50: 0.6225
  avg_hit@1: 0.3491
  avg_hit@5: 0.5273
  avg_mrr: 0.4282
⚠ 5 queries failed. Top errors: {'TimeoutError': 5}

▶ Starting Experiment 35/36: GROUNDER335_MAX3000_MAX20_TOP20_SCO0.85_SUP0.1
Parameters: {'max_candidates_per_symbol': 3000, 'max_answer_candidates': 20, 'top_k_neighbors': 20, 'score_decay': 0.85, 'support_boost': 0.1}
Output Directory: ./output/GROUNDER335_MAX3000_MAX20_TOP20_SCO0.85_SUP0.1


Exp 35 Queries: 100%|██████████| 280/280 [08:49<00:00,  1.89s/query] 


[SAVED] Aggregate results saved to ./output/GROUNDER335_MAX3000_MAX20_TOP20_SCO0.85_SUP0.1/aggregate_results.csv

Aggregate Metrics:
  total_queries: 275
  avg_total_answers: 2.5200
  avg_retrieved_count: 1.4255
  avg_missed_count: 1.0945
  avg_recall@20: 0.6261
  avg_recall@50: 0.6261
  avg_hit@1: 0.3455
  avg_hit@5: 0.5200
  avg_mrr: 0.4211
⚠ 5 queries failed. Top errors: {'TimeoutError': 5}

▶ Starting Experiment 36/36: GROUNDER336_MAX3000_MAX20_TOP20_SCO0.85_SUP0.15
Parameters: {'max_candidates_per_symbol': 3000, 'max_answer_candidates': 20, 'top_k_neighbors': 20, 'score_decay': 0.85, 'support_boost': 0.15}
Output Directory: ./output/GROUNDER336_MAX3000_MAX20_TOP20_SCO0.85_SUP0.15


Exp 36 Queries: 100%|██████████| 280/280 [08:42<00:00,  1.87s/query] 

[SAVED] Aggregate results saved to ./output/GROUNDER336_MAX3000_MAX20_TOP20_SCO0.85_SUP0.15/aggregate_results.csv

Aggregate Metrics:
  total_queries: 275
  avg_total_answers: 2.5200
  avg_retrieved_count: 1.4255
  avg_missed_count: 1.0945
  avg_recall@20: 0.6261
  avg_recall@50: 0.6261
  avg_hit@1: 0.3418
  avg_hit@5: 0.5091
  avg_mrr: 0.4150
⚠ 5 queries failed. Top errors: {'TimeoutError': 5}

🎉 All experiments completed.
